# Musicm8 — AI Producer + Reference Library

This is the primary workflow.

**Your songs → stems + MIDI + EnCodec links + spectral fingerprints → local pretrained AI producer brain → coherent song plan → editable MIDI → Musicm8's own synth/FX engine → audio stems + master**

Change only the `IDEA` line in the big cell to describe what you want. The first run may be slower while Demucs analyses the references and the local AI model downloads. Both the reference analysis and AI model cache are kept in Google Drive.

The AI is the **producer brain**: it understands the text idea, reads summaries of your reference songs, chooses useful reference traits, decides BPM/key/sections/chord degrees/groove and sound-design controls. The deterministic music engine then turns that plan into valid musical material instead of free-running random notes.

> Colab still has to allocate a GPU. If CUDA is unavailable, choose **Runtime → Change runtime type → GPU**, then run the same cell again.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK AI PRODUCER
# references -> AI plan -> coherent MIDI -> own synth/FX -> master
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

# ---------------------------------------------
# EDIT THIS
# ---------------------------------------------
IDEA = "dark UK garage track, 132 BPM feel, emotional chords, deep moving bass, spacious pads"
BARS = 32
SEED = 42

# Small local pretrained language model used as producer brain.
# It is cached in Drive, so later sessions do not need to download it again.
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
WORK = ROOT / "work"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"

AUDIO.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

# Keep the local AI model cache across Colab resets.
HF_CACHE = WORK / "hf_cache"
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Always refresh Musicm8 itself from GitHub.
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)

print("\nInstalling Musicm8 AI-producer dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"], check=True)

import torch
print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU connected. Choose Runtime → Change runtime type → GPU, "
        "then run THIS SAME CELL again."
    )
print("GPU:", torch.cuda.get_device_name(0))

audio_files = [
    p for p in AUDIO.rglob("*")
    if p.is_file() and p.suffix.lower() in {".wav",".mp3",".flac",".m4a",".aac",".ogg",".opus"}
]
print("Reference songs:", len(audio_files))
if not audio_files:
    raise FileNotFoundError(f"Put your reference songs in {AUDIO}")

cmd = [
    sys.executable, "-u", "ai_producer_workflow.py",
    "--root", str(ROOT),
    "--repo", str(REPO),
    "--idea", IDEA,
    "--bars", str(BARS),
    "--seed", str(SEED),
    "--ai-model", AI_MODEL,
]

print("\n🚀 Starting Musicm8 AI producer...")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
MASTER = PROJECT / "master.wav"
PLAN = PROJECT / "plan.json"
MIDI = PROJECT / "arrangement.mid"
LIBRARY = WORK / "reference_library/library.json"

print("\n✅ MUSICM8 AI PROJECT")
print("Idea       :", IDEA)
print("Plan       :", PLAN)
print("References :", LIBRARY)
print("MIDI       :", MIDI)
print("MIDI stems :", PROJECT / "midi_stems")
print("Audio stems:", PROJECT / "audio_stems")
print("Synth patch:", PROJECT / "synth_patches.json")
print("Master     :", MASTER)

from IPython.display import Audio, display
if MASTER.exists():
    print("\n▶️ MUSICM8 INTERNAL SYNTH/FX MASTER")
    display(Audio(str(MASTER)))
else:
    print("⚠️ Master WAV was not produced.")

print("""
HOW THIS WORKS
• The 11 songs remain your reference library, not one waveform-training target.
• Demucs stems + extracted MIDI describe arrangement and performance.
• Spectral fingerprints describe sub/bass/mid/high energy, brightness, transients and dynamics.
• Existing EnCodec tokens are linked into the reference library for future neural resynthesis.
• The local pretrained AI reads those reference summaries and your IDEA, then writes plan.json.
• The music engine enforces key/chords/repeated motifs/groove so output is not random MIDI.
• Musicm8's own synth creates drums/bass/chords/melody and applies FX.
• MIDI, synth patches and audio stems stay editable.
""")


## Optional: inspect saved AI project files


In [ ]:
from pathlib import Path
root = Path("/content/drive/MyDrive/Musicm8/work")
project = root / "ai_projects/latest"
print("Reference library:", root / "reference_library/library.json")
print("Latest AI project:", project)
for p in sorted(project.rglob("*")) if project.exists() else []:
    if p.is_file():
        print(" -", p)
